                                Statistical Analysis & Predictive Modelling of Particle-Dampers
The objective of this project is to develop a machine learning model to predict the vibration reduction achieved by a particle damper based on its parameters.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import GradientBoostingRegressor

In [13]:
# Data Loading
df = pd.read_excel("Particle_damping_vibration.xlsx")
df.head()

,Particle_Diameter_m,Particle_Density_kg_m3,Number_of_Particles,Enclosure_Mass_kg,Clearance_m,Restitution_Coefficient,Vibration_Reduction_Percent
0,0.001972,7597.418515,195,0.011403,0.004511,0.513661,59.721813
1,0.001931,7672.918942,164,0.014928,0.002897,0.445797,57.548744
2,0.001907,7517.400086,190,0.016012,0.001092,0.496480,56.549816
3,0.001930,6455.848603,187,0.028741,0.001665,0.334337,54.776176
4,0.001923,7462.350927,177,0.041064,0.004745,0.364022,53.597649


In [14]:
# Engineered Features
df["Total_Particle_Mass_kg"] = (
    df["Number_of_Particles"]
    * (np.pi / 6)
    * df["Particle_Diameter_m"]**3
    * df["Particle_Density_kg_m3"]
)
df["Particle_to_Enclosure_Mass_Ratio"] = (
    df["Total_Particle_Mass_kg"] / df["Enclosure_Mass_kg"]
)

In [15]:
# Defining features and target
features = [
    "Particle_Diameter_m",
    "Particle_Density_kg_m3",
    "Number_of_Particles",
    "Enclosure_Mass_kg",
    "Clearance_m",
    "Restitution_Coefficient",
    "Total_Particle_Mass_kg",
    "Particle_to_Enclosure_Mass_Ratio"
]

target = "Vibration_Reduction_Percent"

X = df[features]
y = df[target]

In [16]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (4000, 8)
Testing set: (1000, 8)


                                          Baseline Model:Linear Regression

In [17]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_lr = linear_model.predict(X_test)

In [18]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = mean_squared_error(y_test, y_pred_lr) ** 0.5
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance")
print("MAE :", round(mae_lr, 4))
print("RMSE:", round(rmse_lr, 4))
print("R²  :", round(r2_lr, 4))

Linear Regression Performance
MAE : 2.67
RMSE: 3.6013
R²  : 0.9011


                                            Random Forest Regression

In [19]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [20]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = mean_squared_error(y_test, y_pred_rf) ** 0.5
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Performance")
print(f"MAE : {mae_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"R²  : {r2_rf:.4f}")

Random Forest Performance
MAE : 0.4798
RMSE: 0.8532
R²  : 0.9945


In [21]:
print("Training R²:", rf_model.score(X_train, y_train))
print("Testing R² :", rf_model.score(X_test, y_test))

Training R²: 0.9991183118312018
Testing R² : 0.994450440849847


In [22]:
# Feature Importance by Random Forest
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance

,Feature,Importance
6,Total_Particle_Mass_kg,0.800344
5,Restitution_Coefficient,0.186776
7,Particle_to_Enclosure_Mass_Ratio,0.008126
3,Enclosure_Mass_kg,0.001665
4,Clearance_m,0.000815
1,Particle_Density_kg_m3,0.000786
0,Particle_Diameter_m,0.000750
2,Number_of_Particles,0.000739


In [ ]:
# 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_r2_rf = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

cv_mae_rf = -cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=kf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

cv_rmse_rf = -cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("Cross-Validation Results")
print(f"R²   : {cv_r2_rf.mean():.4f} ± {cv_r2.std():.4f}")
print(f"MAE  : {cv_mae_rf.mean():.4f} ± {cv_mae.std():.4f}")
print(f"RMSE : {cv_rmse_rf.mean():.4f} ± {cv_rmse.std():.4f}")

                                                   Gradient Boosting Model

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
gb_model.fit(X_train, y_train)

y_pred_gb = gb_model.predict(X_test)

In [ ]:
mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = mean_squared_error(y_test, y_pred_gb) ** 0.5
r2_gb = r2_score(y_test, y_pred_gb)

print("Gradient Boosting Performance")
print(f"MAE : {mae_gb:.4f}")
print(f"RMSE: {rmse_gb:.4f}")
print(f"R²  : {r2_gb:.4f}")

In [ ]:
# Feature Importance by Gradient Boosting
feature_importance_gb = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": gb_model.feature_importances_
}).sort_values("Importance", ascending=False)

print(feature_importance_gb.round(6))

In [ ]:
# 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_r2_gb = cross_val_score(
    gb_model,
    X_train,
    y_train,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

cv_mae_gb = -cross_val_score(
    gb_model,
    X_train,
    y_train,
    cv=kf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

cv_rmse_gb = -cross_val_score(
    gb_model,
    X_train,
    y_train,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("Gradient Boosting Cross-Validation")
print(f"R²   : {cv_r2_gb.mean():.4f} ± {cv_r2_gb.std():.4f}")
print(f"MAE  : {cv_mae_gb.mean():.4f} ± {cv_mae_gb.std():.4f}")
print(f"RMSE : {cv_rmse_gb.mean():.4f} ± {cv_rmse_gb.std():.4f}")

                                               Model Comparison

In [ ]:
model_comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Gradient Boosting"],
    "MAE": [mae_lr, cv_mae_rf.mean(), cv_mae_gb.mean()],
    "RMSE": [rmse_lr, cv_rmse_rf.mean(), cv_rmse_gb.mean()],
    "R²": [r2_lr, cv_r2_rf.mean(), cv_r2_gb.mean()]
})

model_comparison.round(4)

                                            Findings and Conclusions
- Machine learning models successfully captured the relationship between particle-damper parameters and vibration reduction, with Gradient Boosting and Random Forest achieving very high predictive performance.
- Gradient Boosting performed slightly better than Random Forest, achieving test (R^2 = 0.9956), MAE = 0.446, and RMSE = 0.759.
- Feature importance consistently identified Total Particle Mass as the dominant predictor, followed by the Restitution Coefficient, showing that particle loading and energy dissipation characteristics strongly influence vibration reduction.
- Linear Regression provided a strong baseline with (R^2 = 0.9011), but its lower performance compared with tree-based models suggests that linear relationships alone cannot fully explain vibration reduction.
- Overall, the results demonstrate that feature engineering combined with nonlinear ML models provides an effective approach for predicting vibration reduction.